In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import eigvals
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
#from tqdm import tqdm

In [ ]:
# ----- build the weight matrix for analyzing M-WTA -----

def weight_matrix(M, W1, W2, W3):
    
    W = np.zeros((M, M))
    
    # self excitation
    for i in range(M):
        W[i, i] += W1 
    
    # global inhibition among these neurons
    W -= W2
    
    # nearest-neighbor excitation (linear chain)
    for i in range(M - 1):
        W[i, i+1] += W3
        W[i+1, i] += W3
    
    return W

In [ ]:
def check_MWTA_stability(M, W1, W2, W3, I_ext, tol=1e-12):

    W = weight_matrix(M, W1, W2, W3)

    # ----- Consistency Condition -----
    # Consistency for M-WTA at steady state requires all x1,x2,...,xm > 0
    A = np.eye(M) - W
    b = I_ext * np.ones(M)
      
    try:
        x_s = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return False 
    
    if np.min(x_s) <= tol:
        return False

    # ---- Neighboring neurons suppresion condition ----
    # the neighboring neurons of the active neurons are suppressed (total inputs to these neurons are negative) at steady state
    x_left = -W2*np.sum(x_s) + W3*x_s[0] + I_ext
    x_right = -W2*np.sum(x_s) + W3*x_s[-1] + I_ext
    
    if (x_left > tol) or (x_right > tol) :
        return False
        
    # ----- Stablity Condition -----
    # Eigen values should be positive integers and maximum eigen value should be less than 1
    eigenvalues = np.linalg.eigvals(W)

    # Check for complex numbers
    if not np.all(np.isreal(eigenvalues)):
        return False
    
        
    # Check for the upper bound (must be <= 1)
    if np.max(eigenvalues) >= 1:
        return False


    return True

#       check_MWTA_stability(M, W1, W2, W3, I_ext, tol=1e-12):
M, W1, W2, W3, I_ext = 1, 0, 3, 1, 2
print("stable:", check_MWTA_stability(M, W1, W2, W3, I_ext))
print("-----")
print("stable M+1:", check_MWTA_stability(M+1, W1, W2, W3, I_ext))


In [ ]:
# Parameter grid
W1_vals = np.arange(0, 5.01, 0.05)
W3_vals = np.arange(0, 5.01, 0.05)

# values
M = 5
W1 = 2.0
W2 = 2.0
W3 = 1.0
I_ext = 2.0

phase = np.zeros((len(W1_vals), len(W3_vals)))

for i, W1 in enumerate(W1_vals):
    for j, W3 in enumerate(W3_vals):
        
        M_stable = check_MWTA_stability(M, W1, W2, W3, I_ext)
        M1_stable = check_MWTA_stability(M+1, W1, W2, W3, I_ext)
        
        # M-WTA is stable and (M+1)-WTA is unstable
        phase[i, j] = int(M_stable and not M1_stable)


# plot
plt.figure(figsize=(6,5))
plt.imshow(
    phase.T,
    extent=[0,5,0,5],
    origin='lower',
    aspect='auto'
)
plt.xlabel("W1")
plt.ylabel("W3")
plt.title(f"{M}-WTA Stable & Consistent Region")
plt.colorbar(label="1 = valid (green), 0 = invalid (red)")
plt.show()

## Time evolution

In [ ]:
# -----------------------------
# Current injection model (simulate x(t))
# -----------------------------
def relu(z):
    return np.maximum(0.0, z)

def H(z):
    return (z > 0).astype(float)

# constants
N = 50
tau = 5e-3          # s
dt  = 1e-3          # s
T   = 2.0           # s
steps = int(T/dt)

# parameters
W1, W2, W3, W4, W5 = 0.5, 1, 0.5, 0.05, 1.0
I = 2.0
omega = 10.0        

# start with random x in [0, 1]
rng = np.random.default_rng(0)
x = rng.uniform(0.0, 1.0, size=N)

stride = 2   # store frames for animation (downsample for speed)
Xs = []      # stores the full activity x at all time series

for k in range(steps):
    # current injection model r_k-1, l_k+1
    r = relu(H(x - 0.1) - 1.0 + W4 * relu(omega))
    l = relu(H(x - 0.1) - 1.0 + W4 * relu(-omega))

    # ring terms
    x_left  = np.roll(x, 1)
    x_right = np.roll(x, -1)
    
    # noise term
    sigma = 0.1
    I_ext = relu(I + rng.normal(0, sigma, size=N))  

    total = (W1 * x) + -W2 * np.sum(x) + (W3 * (x_left + x_right)) + W5 * (np.roll(r, 1) + np.roll(l, -1)) + I_ext
    x = x + dt * ((-x + relu(total)) / tau)

    if k % stride == 0:
        Xs.append(x.copy())

Xs = np.array(Xs)  # shape: (n_frames, N)

# -----------------------------
# Bar-plot animation of x(t)
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(np.arange(N), Xs[0])
ax.set_xlim(-0.5, N - 0.5)
ax.set_ylim(0, max(1.0, Xs.max()*1.05))
ax.set_xlabel("Neuron index k")
ax.set_ylabel("Firing rate x_k")
title = ax.set_title("")

def update(frame):
    y = Xs[frame]
    for b, yi in zip(bars, y):
        b.set_height(yi)
    title.set_text(f"t = {frame*stride*dt:.3f} s")
    return (*bars, title)

anim = FuncAnimation(fig, update, frames=len(Xs), interval=30, blit=True)
plt.close(fig)

HTML(anim.to_jshtml())


In [ ]:
k = np.arange(N)
theta = 360.0 * k / N   # shape (N,)
theta_rad = np.deg2rad(theta)

cos_part = Xs @ np.cos(theta_rad)
sin_part = Xs @ np.sin(theta_rad)

avg_theta = np.rad2deg(np.arctan2(sin_part, cos_part)) % 360

plt.figure(figsize=(7,4))
plt.plot(avg_theta)
plt.xlabel("Frame")
plt.ylabel("Average θ (deg)")
plt.title("Average Head Direction vs Frame")
plt.show()

In [ ]:
dt = 0.001
time = np.arange(len(avg_theta)) * dt

omega_vals = np.ones_like(time) * omega  # example 50 deg/s
# omega(t) evaluated at each time

theta_true = np.cumsum(omega_vals) * dt
theta_true = theta_true % 360

plt.figure(figsize=(8,4))
plt.plot(time, avg_theta, label="Decoded θ")
plt.plot(time, theta_true, '--', label="True θ")
plt.xlabel("Time (s)")
plt.ylabel("Head Direction (deg)")
plt.legend()
plt.show()


# Sinosoidal omega

In [ ]:
# -----------------------------
# Current injection model (simulate x(t))
# -----------------------------
def relu(z):
    return np.maximum(0.0, z)

def H(z):
    return (z > 0).astype(float)

# constants
N = 50
tau = 5e-3          # s
dt  = 1e-3          # s
T   = 2.0           # s
steps = int(T/dt)

# parameters
W1, W2, W3, W4, W5 = 0.5, 0.5, 0.5, 0.05, 1.0
I = 2.0
omega_0 = 200.0        
omega_1 = 1
omega = 0

# start with random x in [0, 1]
rng = np.random.default_rng(0)
x = rng.uniform(0.0, 1.0, size=N)

stride = 2   # store frames for animation (downsample for speed)
Xs = []      # stores the full activity x at all time series

for k in range(steps):
    # current injection model r_k-1, l_k+1
    r = relu(H(x - 0.1) - 1.0 + W4 * relu(omega))
    l = relu(H(x - 0.1) - 1.0 + W4 * relu(-omega))

    # ring terms
    x_left  = np.roll(x, 1)
    x_right = np.roll(x, -1)
    
    # noise term
    sigma = 0.1
    I_ext = relu(I + rng.normal(0, sigma, size=N))  

    total = (W1 * x) + -W2 * np.sum(x) + (W3 * (x_left + x_right)) + W5 * (np.roll(r, 1) + np.roll(l, -1)) + I_ext
    x = x + dt * ((-x + relu(total)) / tau)
    omega = omega_0 * np.sin(omega_1 * k)

    if k % stride == 0:
        Xs.append(x.copy())

Xs = np.array(Xs)  # shape: (n_frames, N)

# -----------------------------
# Bar-plot animation of x(t)
# -----------------------------
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(np.arange(N), Xs[0])
ax.set_xlim(-0.5, N - 0.5)
ax.set_ylim(0, max(1.0, Xs.max()*1.05))
ax.set_xlabel("Neuron index k")
ax.set_ylabel("Firing rate x_k")
title = ax.set_title("")

def update(frame):
    y = Xs[frame]
    for b, yi in zip(bars, y):
        b.set_height(yi)
    title.set_text(f"t = {frame*stride*dt:.3f} s")
    return (*bars, title)

anim = FuncAnimation(fig, update, frames=len(Xs), interval=30, blit=True)
plt.close(fig)

HTML(anim.to_jshtml())


In [ ]:
dt = 0.001
time = np.arange(len(avg_theta)) * dt

omega_0 = 200.0        # deg/s
Omega = 2*np.pi*2    # 1 Hz

omega_vals = omega_0 * np.sin(Omega * time)

theta_true = np.cumsum(omega_vals) * dt % 360

plt.figure(figsize=(8,4))
plt.plot(time, avg_theta, label="Decoded θ")
plt.plot(time, theta_true, '--', label="True θ")
plt.xlabel("Time (s)")
plt.ylabel("Head Direction (deg)")
plt.legend()
plt.show()
